In [25]:
from __future__ import print_function, unicode_literals, absolute_import, division
import os
import sys
import numpy as np
# matplotlib.rcParams["image.interpolation"] = None

from glob import glob
from tqdm import tqdm
from tifffile import imread
from csbdeep.utils import Path, normalize

from stardist import fill_label_holes, random_label_cmap, calculate_extents, gputools_available
from stardist.models import Config2D, StarDist2D
import argparse
import yaml
import shutil
import pandas as pd
from typing import Optional

config = {}


save_dir = config.get("save_dir", None)
model_name = config.get("model_name", None)
checkpoint_path = config.get("continue_training_from_checkpoint", None)
pretrained_model = config.get("pretrained_model", None)
n_rays = config.get("n_rays", 32)
panoptic = config.get("panoptic", True)
n_classes = config.get("n_classes", None)
max_epochs = config.get("max_epochs", 400)
steps_per_epoch = config.get("steps_per_epoch", 100)
patch_size = config.get("patch_size", 512)
train_val_split_ratio = config.get("train_val_split_ratio", 0.2)
downsample_factor = config.get("downsample_factor", 1)
channels_to_segment = config.get("channels_to_segment", [0])
use_gpu = config.get("use_gpu", True)

conf = Config2D (
    n_rays       = n_rays,
    grid         = [1,1],
    use_gpu      = use_gpu,
    n_channel_in = 1,
    train_epochs = max_epochs,
    train_steps_per_epoch = steps_per_epoch,
    train_patch_size = 512,
)

conf.n_classes = 10
model = StarDist2D(conf, name='emr1_60x_panoptic_seg_only', basedir='/mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/models/stardist/')

keras_model = model.keras_model

keras_model.get_layer(name='features_class').trainable = False
keras_model.get_layer(name='prob_class').trainable = False


Using default values: prob_thresh=0.5, nms_thresh=0.4.


In [26]:
print(model.keras_model.summary())

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, None,      │          0 │ -                 │
│                     │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_0_no_0   │ (None, None,      │        320 │ input[0][0]       │
│ (Conv2D)            │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_0_no_1   │ (None, None,      │      9,248 │ down_level_0_no_… │
│ (Conv2D)            │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_0               │ (None, None,      │          0 │ down_level_0_no_… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_1_no_0   │ (None, None,      │     18,496 │ max_0[0][0]       │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_1_no_1   │ (None, None,      │     36,928 │ down_level_1_no_… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_1               │ (None, None,      │          0 │ down_level_1_no_… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_2_no_0   │ (None, None,      │     73,856 │ max_1[0][0]       │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_2_no_1   │ (None, None,      │    147,584 │ down_level_2_no_… │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_2               │ (None, None,      │          0 │ down_level_2_no_… │
│ (MaxPooling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_0 (Conv2D)   │ (None, None,      │    295,168 │ max_2[0][0]       │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_2 (Conv2D)   │ (None, None,      │    295,040 │ middle_0[0][0]    │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_45    │ (None, None,      │          0 │ middle_2[0][0]    │
│ (UpSampling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_45      │ (None, None,      │          0 │ up_sampling2d_45… │
│ (Concatenate)       │ None, 256)        │            │ down_level_2_no_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_level_2_no_0     │ (None, None,      │    295,040 │ concatenate_45[0… │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_level_2_no_2     │ (None, None,      │     73,792 │ up_level_2_no_0[… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_46    │ (None, None,      │          0 │ up_level_2_no_2[

 Total params: 1,445,100 (5.51 MB)

 Trainable params: 1,406,689 (5.37 MB)

 Non-trainable params: 38,411 (150.04 KB)

None


In [27]:
conf = Config2D (
    n_rays       = n_rays,
    grid         = [1,1],
    use_gpu      = use_gpu,
    n_channel_in = 1,
    train_epochs = max_epochs,
    train_steps_per_epoch = steps_per_epoch,
    train_patch_size = 512,
)

conf.n_classes = None
model = StarDist2D(conf, name='emr1_60x_panoptic_seg_only', basedir='/mnt/towbin.data/shared/spsalmon/towbinlab_segmentation_database/models/stardist/')
print(model.keras_model.summary())

Using default values: prob_thresh=0.5, nms_thresh=0.4.


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, None,      │          0 │ -                 │
│                     │ None, 1)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_0_no_0   │ (None, None,      │        320 │ input[0][0]       │
│ (Conv2D)            │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_0_no_1   │ (None, None,      │      9,248 │ down_level_0_no_… │
│ (Conv2D)            │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_0               │ (None, None,      │          0 │ down_level_0_no_… │
│ (MaxPooling2D)      │ None, 32)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_1_no_0   │ (None, None,      │     18,496 │ max_0[0][0]       │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_1_no_1   │ (None, None,      │     36,928 │ down_level_1_no_… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_1               │ (None, None,      │          0 │ down_level_1_no_… │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_2_no_0   │ (None, None,      │     73,856 │ max_1[0][0]       │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ down_level_2_no_1   │ (None, None,      │    147,584 │ down_level_2_no_… │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_2               │ (None, None,      │          0 │ down_level_2_no_… │
│ (MaxPooling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_0 (Conv2D)   │ (None, None,      │    295,168 │ max_2[0][0]       │
│                     │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_2 (Conv2D)   │ (None, None,      │    295,040 │ middle_0[0][0]    │
│                     │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_48    │ (None, None,      │          0 │ middle_2[0][0]    │
│ (UpSampling2D)      │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_48      │ (None, None,      │          0 │ up_sampling2d_48… │
│ (Concatenate)       │ None, 256)        │            │ down_level_2_no_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_level_2_no_0     │ (None, None,      │    295,040 │ concatenate_48[0… │
│ (Conv2D)            │ None, 128)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_level_2_no_2     │ (None, None,      │     73,792 │ up_level_2_no_0[… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_49    │ (None, None,      │          0 │ up_level_2_no_2[

 Total params: 1,406,689 (5.37 MB)

 Trainable params: 1,406,689 (5.37 MB)

 Non-trainable params: 0 (0.00 B)

None
